# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id
record_sets = [r for r in metadata.record_sets]
print(f"Found {len(record_sets)} record set(s):")
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name}, description: {getattr(record_set, 'description', '')}")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect the @ids of the record sets
record_set_ids = [r.id for r in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet: {record_set_id} (rows: {len(records)})")
    else:
        print(f"No records found for RecordSet: {record_set_id}")

if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in first DataFrame ({first_record_set_id}):")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No DataFrames loaded - check dataset availability.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Assume protein DataFrame is the first record set if present
if dataframes:
    df = dataframes[first_record_set_id]
    print(f"Using DataFrame from RecordSet: {first_record_set_id}")

    # List columns to pick numeric and group fields
    print("Available columns:", df.columns.tolist())

    # Try to pick a numeric field, e.g., 'coverage' or 'MW' (molecular weight)
    candidate_numeric_fields = [c for c in df.columns if df[c].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[c])]

    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field}")

        # Set an arbitrary threshold for demonstration
        threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a field with 'type', 'category', or 'sample' in its name, else just the second column (if present)
        candidate_group_field = None
        for col in df.columns:
            if any(s in col.lower() for s in ['type', 'category', 'sample', 'group']):
                candidate_group_field = col
                break
        if candidate_group_field is None and len(df.columns) > 1:
            candidate_group_field = df.columns[1]

        group_field = candidate_group_field
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No appropriate group field found for grouping.")
    else:
        print("No numeric field found to perform EDA.")
else:
    print("No DataFrame available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and candidate_numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset offers a comprehensive analysis of protein abundance and modifications measured by mass spectrometry in human mast cell extracellular vesicle samples.
* Using `mlcroissant`, we loaded dataset record sets and fields programmatically using their `@id` values for reproducibility.
* The notebook demonstrated extraction, filtering, normalization, grouping, and visualization of quantitative protein fields.
* These steps are adaptable using other `@id` values and fields for further hypothesis-driven exploration.